Streamlit インストール（Colab or VSCode）

In [1]:
pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.2 MB/s eta 0:00:00


In [2]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

def extract_landmarks_from_image(path):
    img = cv2.imread(path)
    if img is None:
        return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)

    if not result.multi_hand_landmarks:
        return None

    data = []
    lm = result.multi_hand_landmarks[0]
    for p in lm.landmark:
        data.extend([p.x, p.y, p.z])

    return data


ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
code = """
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

def extract_landmarks_from_image(path):
    img = cv2.imread(path)
    if img is None:
        return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)

    if not result.multi_hand_landmarks:
        return None

    data = []
    lm = result.multi_hand_landmarks[0]
    for p in lm.landmark:
        data.extend([p.x, p.y, p.z])

    return data
"""

with open("/content/drive/MyDrive/sign_letter_ai/sign_app /landmarks_extract.py", "w") as f:
    f.write(code)



In [3]:
app_code = """
import streamlit as st
import pickle
import numpy as np
from landmarks_extract import extract_landmarks_from_image

st.title("指文字（あ行）認識アプリ")

# 学習済みモデルの読み込み
with open("ai_model.pkl", "rb") as f:
    model = pickle.load(f)

uploaded = st.file_uploader(
    "画像をアップロードしてください（あ・い・う・え・お）",
    type=["jpg", "jpeg", "png", "JPG"]
)

if uploaded is not None:
    temp_path = "temp.jpg"
    with open(temp_path, "wb") as f:
        f.write(uploaded.getvalue())

    feature = extract_landmarks_from_image(temp_path)

    if feature is None:
        st.error("手の検出に失敗しました。画像を変えてみてください。")
    else:
        pred = model.predict(np.array(feature).reshape(1, -1))
        st.success(f"判定結果： {pred[0]}")
        st.image(uploaded)
"""

with open("/content/drive/MyDrive/sign_letter_ai/sign_app /app.py", "w") as f:
    f.write(app_code)
